# Caso de Uso: Propension de Venta

**Curso:** MLOps | Maestria en Ciencia de Datos | URP 2026-I  
**Docente:** MSc. Manasses Antoni Mauricio Condori

---

## Contenido
1. Contexto del problema
2. Carga y exploracion de datos
3. Identificacion de problemas en datos
4. Preprocesamiento: limpieza, imputacion, encoding
5. Division temporal de datos
6. Entrenamiento del modelo baseline (XGBoost)
7. Evaluacion del modelo

## 1. Contexto del problema

**Negocio:** Una entidad financiera desea identificar que clientes tienen mayor
probabilidad de adquirir un producto de tarjeta de credito.

**Objetivo del modelo:**  
Predecir la variable binaria `target` (1 = cliente compro, 0 = no compro) a partir
de variables comportamentales, transaccionales y de riesgo.

**Relevancia para MLOps:**
- El modelo se ejecutara mensualmente sobre ~80 000 clientes.
- Debe ser reproducible, auditable y monitoreable en produccion.
- Un mal preprocesamiento es la principal causa de degradacion de modelos en produccion.

## 2. Carga de librerias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)
import xgboost as xgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

## 3. Carga de datos

In [ ]:
DATA_PATH = 'Notebooks basicos/Data_CU_venta.csv'

df_raw = pd.read_csv(DATA_PATH)
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

## 4. Exploracion de datos (EDA)

### 4.1 Dimensiones y tipos de variables

In [ ]:
print('Filas:', df_raw.shape[0], '| Columnas:', df_raw.shape[1])
print()
print('Tipos de datos:')
print(df_raw.dtypes.value_counts())

### 4.2 Balance del target

In [ ]:
target_counts = df_raw['target'].value_counts(normalize=True) * 100
print('Distribucion del target (%):')
print(target_counts.round(2))

fig, ax = plt.subplots(figsize=(5, 3))
target_counts.plot(kind='bar', color=['#145D38', '#9C247E'], ax=ax)
ax.set_title('Distribucion del Target')
ax.set_ylabel('Porcentaje (%)')
ax.set_xticklabels(['No compro (0)', 'Compro (1)'], rotation=0)
plt.tight_layout()
plt.show()

# Dataset fuertemente desbalanceado (~3.8% positivos)
# Usar metricas como AUC-ROC y F1, no accuracy.

### 4.3 Analisis de valores nulos

In [ ]:
def calculo_nan(df, umbral=0.0):
    for c in df.columns:
        porc_nan = df[c].isna().sum() / len(df[c]) * 100
        if porc_nan > umbral:
            print(c, ' ', round(porc_nan, 2))

calculo_nan(df_raw, umbral=20)

In [ ]:
# Resumen tabular — variables con algun nulo
def resumen_nulos(df, umbral=0.0):
    pct_nan = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
    return pct_nan[pct_nan > umbral].rename('pct_nulos').to_frame()

nulos = resumen_nulos(df_raw, umbral=0.0)
print(f'Variables con al menos un nulo: {len(nulos)}')
print()
print('Top 15 variables con mas nulos:')
print(nulos.head(15).round(2))

In [ ]:
# Visualizacion de nulos > 5%
nulos_sig = resumen_nulos(df_raw, umbral=5.0)

if not nulos_sig.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    nulos_sig['pct_nulos'].plot(kind='barh', ax=ax, color='#9C247E')
    ax.axvline(x=80, color='red', linestyle='--', label='Umbral descarte (80%)')
    ax.set_title('Variables con >5% de valores nulos')
    ax.set_xlabel('% Nulos')
    ax.legend()
    plt.tight_layout()
    plt.show()

### 4.4 Variables categoricas detectadas

In [ ]:
vars_categoricas = df_raw.select_dtypes(include='object').columns.tolist()
print('Variables categoricas:', vars_categoricas)
print()
for col in vars_categoricas:
    print(f'{col}: {df_raw[col].nunique()} categorias -> {df_raw[col].unique()[:8]}')

### 4.5 Estadisticas descriptivas

In [ ]:
df_raw.describe().T

## 5. Identificacion de problemas tipicos en datos

| Problema | Variables afectadas | Tratamiento |
|---|---|---|
| Missing values >80% | prm_saltotrdpj03m, prm_saltotrdpj12m, dsv_diasatrrdpj12m, max_pctsalimpago12m, prm_diasatrrdpj03m | Eliminar del dataset |
| Missing values moderados | dsv_saltotppe03m, beta_saltotppe12m, beta_saltotppe06m | Imputar con 0 |
| Variables categoricas | sexo, region, ubigeo_buro, grp_riesgociiu, grp_camptot06m, etc. | LabelEncoder |
| Columnas identificadoras | p_codmes, key_value | Excluir del modelo |
| Data leakage potencial | prob_value | Excluir (score del modelo anterior) |

> **Data Leakage**: `prob_value` es el score del modelo anterior.
> Incluirlo generaria un modelo que copia predicciones previas.

## 6. Preprocesamiento

### 6.1 Definicion de columnas excluidas y columnas a tratar

In [ ]:
# Columnas identificadoras
COLS_EXCLUIR = ['p_codmes', 'key_value', 'prob_value']

# Columnas con >80% de nulos
UMBRAL_DESCARTE = 80.0
nulos_full = resumen_nulos(df_raw, umbral=UMBRAL_DESCARTE)
COLS_ALTO_NULO = nulos_full.index.tolist()
print('Columnas descartadas por alto % de nulos:', COLS_ALTO_NULO)

# Columnas categoricas
COLS_CATEGORICAS = df_raw.select_dtypes(include='object').columns.tolist()
print('Columnas categoricas:', COLS_CATEGORICAS)

### 6.2 Imputacion con cero

Variables numericas con nulos por criterio de negocio (ausencia = 0).

In [ ]:
df = df_raw.copy()

df.flg_saltothip12m = df.flg_saltothip12m.fillna(0)
df.flg_saltotppe12m = df.flg_saltotppe12m.fillna(0)
df.prm_pctsaltototrent12m = df.prm_pctsaltototrent12m.fillna(0)
df.prm_pctsaltotcaja12m = df.prm_pctsaltotcaja12m.fillna(0)
df.ant_saltot24m = df.ant_saltot24m.fillna(0)
df.ant_saltot12m = df.ant_saltot12m.fillna(0)
df.min_difsaltottcr12m = df.min_difsaltottcr12m.fillna(0)
df.num_incrsaldispefe06m = df.num_incrsaldispefe06m.fillna(0)
df.max_difent12m = df.max_difent12m.fillna(0)
df.num_dismsalppecons06m = df.num_dismsalppecons06m.fillna(0)
df.beta_pctusotcr12m = df.beta_pctusotcr12m.fillna(0)
df.prm_pctusosaltottcr03m = df.prm_pctusosaltottcr03m.fillna(0)
df.dsv_saltotppe03m = df.dsv_saltotppe03m.fillna(0)
df.prm_diasatrrdpn12m = df.prm_diasatrrdpn12m.fillna(0)
df.dsv_numentrdlintcr03m = df.dsv_numentrdlintcr03m.fillna(0)
df.rat_disefepnm01 = df.rat_disefepnm01.fillna(0)
df.prm_diasatrrdpn06m = df.prm_diasatrrdpn06m.fillna(0)
df.pct_usotcrm01 = df.pct_usotcrm01.fillna(0)
df.dsv_numentrdlintcr06m = df.dsv_numentrdlintcr06m.fillna(0)
df.beta_saltotppe12m = df.beta_saltotppe12m.fillna(0)
df.prm_entrd03m = df.prm_entrd03m.fillna(0)
df.ctd_entrdm01 = df.ctd_entrdm01.fillna(0)
df.beta_saltotppe06m = df.beta_saltotppe06m.fillna(0)
df.prm_diasatrrd03m = df.prm_diasatrrd03m.fillna(0)
df.prm_saltotrdpj03m = df.prm_saltotrdpj03m.fillna(0)
df.prm_saltotrdpj12m = df.prm_saltotrdpj12m.fillna(0)
df.dsv_diasatrrdpj12m = df.dsv_diasatrrdpj12m.fillna(0)
df.max_pctsalimpago12m = df.max_pctsalimpago12m.fillna(0)
df.prm_diasatrrdpj03m = df.prm_diasatrrdpj03m.fillna(0)
df.ctd_campecstlv06m = df.ctd_campecstlv06m.fillna(0)
df.max_camptot06m = df.max_camptot06m.fillna(0)
df.min_camptot06m = df.min_camptot06m.fillna(0)
df.frc_camptot06m = df.frc_camptot06m.fillna(0)
df.rec_camptot06m = df.rec_camptot06m.fillna(0)
df.ctd_camptot06m = df.ctd_camptot06m.fillna(0)
df.prm_camptot06m = df.prm_camptot06m.fillna(0)
df.max_campecs06m = df.max_campecs06m.fillna(0)
df.min_campecs06m = df.min_campecs06m.fillna(0)
df.frc_campecs06m = df.frc_campecs06m.fillna(0)
df.rec_campecs06m = df.rec_campecs06m.fillna(0)
df.ctd_campecs06m = df.ctd_campecs06m.fillna(0)
df.prm_campecs06m = df.prm_campecs06m.fillna(0)
df.seg_un = df.seg_un.fillna(0)

### 6.3 Imputaciones especiales

In [ ]:
# Imputacion por mediana
df.edad = df.edad.fillna(df.edad.median())

# Imputacion por criterio de negocio
df.ubigeo_buro = df.ubigeo_buro.fillna('Otros')
df.grp_riesgociiu = df.grp_riesgociiu.fillna('grupo_0')
df.grp_camptot06m = df.grp_camptot06m.fillna('Otro')
df.grp_campecs06m = df.grp_campecs06m.fillna('Otro')
df.region = df.region.fillna('Otro')

### 6.4 Consolidacion de categorias

Se agrupan categorias minoritarias para reducir cardinalidad.

In [ ]:
# seg_un: agrupar 0 y 3 como la misma categoria
df['seg_un'] = pd.Series(np.where(df.seg_un.isin([0, 3]), 0, df.seg_un))

# grp_riesgociiu: consolidar grupos minoritarios en grupo_11
df['grp_riesgociiu'] = pd.Series(
    np.where(
        df.grp_riesgociiu.isin(['grupo_2', 'grupo_3', 'grupo_9', 'grupo_8', 'grupo_1']),
        'grupo_11',
        df.grp_riesgociiu,
    )
)

### 6.5 Asignacion de tipo de datos

Parte del principio de reproducibilidad: los tipos deben ser explicitos.

In [ ]:
df.monto = df.monto.astype('float64')
df.target = df.target.astype('int32')
df.ctd_prod_rccsf_m01 = df.ctd_prod_rccsf_m01.astype('int32')
df.prom_salvig_entprinc_pp_rccsf_03m = df.prom_salvig_entprinc_pp_rccsf_03m.astype('float64')
df.max_usotcrrstsf06m = df.max_usotcrrstsf06m.astype('float64')
df.max_usotcrrstsf03m = df.max_usotcrrstsf03m.astype('float64')
df.prm_lintcribksf06m = df.prm_lintcribksf06m.astype('float64')
df.lin_tcribksf03m = df.lin_tcribksf03m.astype('float64')
df.lin_tcribksf06m = df.lin_tcribksf06m.astype('float64')
df.cre_salvig_pp_rccsf_m02 = df.cre_salvig_pp_rccsf_m02.astype('float64')
df.prm_usotcrrstsf06m = df.prm_usotcrrstsf06m.astype('float64')
df.prm_lintcribksf03m = df.prm_lintcribksf03m.astype('float64')
df.prm_usotcrrstsf03m = df.prm_usotcrrstsf03m.astype('float64')
df.ratuso_tcrrstsf_m13 = df.ratuso_tcrrstsf_m13.astype('float64')
df.promctdprodrccsf3m = df.promctdprodrccsf3m.astype('float64')
df.ratpct_saldopprcc_m13 = df.ratpct_saldopprcc_m13.astype('float64')
df.promctdprodrccsf6m = df.promctdprodrccsf6m.astype('float64')
df.ctd_actrccsf_6m = df.ctd_actrccsf_6m.astype('int32')
df.ctd_flgact_rccsf_m01 = df.ctd_flgact_rccsf_m01.astype('int32')
df.sld_ep1ppeallsfm01 = df.sld_ep1ppeallsfm01.astype('float64')
df.prm_sldvigrstsf12m = df.prm_sldvigrstsf12m.astype('float64')
df.sldtot_tcrsrcf = df.sldtot_tcrsrcf.astype('float64')
df.sldvig_tcrsrcf = df.sldvig_tcrsrcf.astype('float64')
df.flg_lintcrripsaga = df.flg_lintcrripsaga.astype('int32')
df.flg_svtcrsrcf = df.flg_svtcrsrcf.astype('int32')
df.flg_sdtcrripsaga = df.flg_sdtcrripsaga.astype('int32')
df.flg_sttcrsrcf = df.flg_sttcrsrcf.astype('int32')
df.flg_svltcrsrcf = df.flg_svltcrsrcf.astype('int32')
df.max_camptottlv06m = df.max_camptottlv06m.astype('int32')
df.min_camptottlv06m = df.min_camptottlv06m.astype('int32')
df.frc_camptottlv06m = df.frc_camptottlv06m.astype('int32')
df.rec_camptottlv06m = df.rec_camptottlv06m.astype('int32')
df.grp_camptottlv06m = df.grp_camptottlv06m.astype('int32')
df.ctd_camptottlv06m = df.ctd_camptottlv06m.astype('int32')
df.prm_camptottlv06m = df.prm_camptottlv06m.astype('float64')
df.max_campecstlv06m = df.max_campecstlv06m.astype('int32')
df.min_campecstlv06m = df.min_campecstlv06m.astype('int32')
df.frc_campecstlv06m = df.frc_campecstlv06m.astype('int32')
df.rec_campecstlv06m = df.rec_campecstlv06m.astype('int32')
df.grp_campecstlv06m = df.grp_campecstlv06m.astype('int32')
df.ctd_campecstlv06m = df.ctd_campecstlv06m.astype('int32')
df.prm_campecstlv06m = df.prm_campecstlv06m.astype('float64')
df.max_camptot06m = df.max_camptot06m.astype('int32')
df.min_camptot06m = df.min_camptot06m.astype('int32')
df.frc_camptot06m = df.frc_camptot06m.astype('int32')
df.rec_camptot06m = df.rec_camptot06m.astype('int32')
df.grp_camptot06m = df.grp_camptot06m.astype('int32')
df.ctd_camptot06m = df.ctd_camptot06m.astype('int32')
df.prm_camptot06m = df.prm_camptot06m.astype('float64')
df.max_campecs06m = df.max_campecs06m.astype('int32')
df.min_campecs06m = df.min_campecs06m.astype('int32')
df.frc_campecs06m = df.frc_campecs06m.astype('int32')
df.rec_campecs06m = df.rec_campecs06m.astype('int32')
df.grp_campecs06m = df.grp_campecs06m.astype('int32')
df.ctd_campecs06m = df.ctd_campecs06m.astype('int32')
df.prm_campecs06m = df.prm_campecs06m.astype('float64')

In [ ]:
# Funcion auxiliar para convertir listas de columnas al mismo tipo
def fun_conv(df, columns, dtype):
    for c in columns:
        if c in df.columns:
            df[c] = df[c].astype(dtype)

# Verificar nulos restantes tras astype
print('Nulos restantes:', df.isna().sum().sum())
df.describe().T

### 6.6 Encoding de variables categoricas

LabelEncoder por columna. Los encoders se guardan para reutilizarlos en inferencia.

In [ ]:
df['ubigeo_buro'].unique()

In [ ]:
features_encoder = [
    'grp_camptottlv06m', 'grp_campecstlv06m',
    'grp_camptot06m', 'grp_campecs06m',
    'region', 'grp_riesgociiu', 'ubigeo_buro',
]
encoders = {}
for col in features_encoder:
    print(col)
    le = LabelEncoder()
    le.fit(df[col])
    df[col] = le.transform(df[col])
    encoders[col] = le

# Monitorear: si aparecen nuevas categorias en produccion el transform fallara.
# Solucion: handle_unknown o reentrenar el encoder.

### 6.7 Pipeline de preprocesamiento (version funcional)

Las mismas operaciones anteriores encapsuladas en funciones reutilizables.

In [ ]:
def eliminar_columnas(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Elimina columnas si existen."""
    return df.drop(columns=[c for c in cols if c in df.columns])


def imputar_nulos_numericos(df: pd.DataFrame, cols: list, valor: float = 0.0) -> pd.DataFrame:
    """Imputa nulos numericos con valor fijo."""
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = df[col].fillna(valor)
    return df


def imputar_nulos_categoricos(df: pd.DataFrame, cols: list, valor: str = 'Otros') -> pd.DataFrame:
    """Imputa nulos categoricos con una categoria por defecto."""
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = df[col].fillna(valor)
    return df


def consolidar_categorias(df: pd.DataFrame, col: str, cats: list, nueva: str) -> pd.DataFrame:
    """Agrupa categorias minoritarias bajo una sola etiqueta."""
    df = df.copy()
    df[col] = np.where(df[col].isin(cats), nueva, df[col])
    return df


def aplicar_label_encoding(df: pd.DataFrame, cols: list) -> tuple:
    """Aplica LabelEncoder. Retorna df y dict de encoders."""
    df = df.copy()
    enc = {}
    for col in cols:
        if col in df.columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            enc[col] = le
    return df, enc

In [ ]:
df_proc = df_raw.copy()

# 1. Eliminar columnas no utiles
df_proc = eliminar_columnas(df_proc, COLS_EXCLUIR + COLS_ALTO_NULO)
print(f'Shape tras eliminar columnas: {df_proc.shape}')

# 2. Imputar numericos
cols_num_nan = (df_proc.select_dtypes(include=[np.number])
                .columns[df_proc.select_dtypes(include=[np.number]).isna().any()]
                .tolist())
df_proc = imputar_nulos_numericos(df_proc, cols_num_nan, valor=0.0)

# 3. Imputar categoricos
df_proc = imputar_nulos_categoricos(df_proc, COLS_CATEGORICAS, valor='Otros')

# 4. Imputacion especial por mediana
if 'edad' in df_proc.columns:
    df_proc['edad'] = df_proc['edad'].fillna(df_proc['edad'].median())

# 5. Consolidar categorias
if 'grp_riesgociiu' in df_proc.columns:
    df_proc = consolidar_categorias(
        df_proc, 'grp_riesgociiu',
        ['grupo_2', 'grupo_3', 'grupo_9', 'grupo_8', 'grupo_1'], 'grupo_11'
    )
if 'seg_un' in df_proc.columns:
    df_proc['seg_un'] = pd.Series(np.where(df_proc['seg_un'].isin([0, 3]), 0, df_proc['seg_un']))

# 6. Encoding
cols_cat_final = df_proc.select_dtypes(include='object').columns.tolist()
df_proc, encoders_proc = aplicar_label_encoding(df_proc, cols_cat_final)

print(f'Shape final: {df_proc.shape}')
print(f'Nulos restantes: {df_proc.isna().sum().sum()}')
df_proc.head(3)

## 7. Division temporal de datos

En proyectos financieros se usa division temporal para respetar la causalidad.
El modelo nunca puede ver el futuro.

- **Entrenamiento:** meses 201909, 201910, 201911
- **Validacion:** mes 201912 (el mas reciente)

In [ ]:
MES_VALIDACION = 201912.0

# Recuperar p_codmes del raw para la particion
df_proc['p_codmes_orig'] = df_raw['p_codmes'].values

df_val_temp = df_proc[df_proc['p_codmes_orig'] == MES_VALIDACION].drop(columns=['p_codmes_orig'])
df_main    = df_proc[df_proc['p_codmes_orig'] != MES_VALIDACION].drop(columns=['p_codmes_orig'])

print(f'Train+Test: {df_main.shape} | Validacion temporal: {df_val_temp.shape}')

TARGET_COL   = 'target'
FEATURE_COLS = [c for c in df_main.columns if c != TARGET_COL]

X_main = df_main[FEATURE_COLS]
y_main = df_main[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X_main, y_main, test_size=0.30, random_state=123, stratify=y_main
)

X_val = df_val_temp[FEATURE_COLS]
y_val = df_val_temp[TARGET_COL]

print(f'X_train: {X_train.shape} | X_test: {X_test.shape} | X_val: {X_val.shape}')
print(f'% positivos train: {y_train.mean()*100:.2f}% | test: {y_test.mean()*100:.2f}% | val: {y_val.mean()*100:.2f}%')

## 8. Entrenamiento del modelo baseline (XGBoost)

Un modelo dummy (sin configuracion particular) establece la baseline de performance
y da una primera idea de la importancia de variables.

In [ ]:
ratio_desbalance = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Ratio desbalance (neg/pos): {ratio_desbalance:.2f}')

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=ratio_desbalance,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)
print('Modelo entrenado.')

## 9. Evaluacion del modelo

In [ ]:
def evaluar_modelo(modelo, X: pd.DataFrame, y: pd.Series, nombre: str) -> dict:
    """Calcula y muestra metricas de clasificacion."""
    y_pred  = modelo.predict(X)
    y_proba = modelo.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, y_proba)
    print(f'\n{"="*50}')
    print(f'  Conjunto: {nombre}')
    print(f'{"="*50}')
    print(f'  AUC-ROC: {auc:.4f}')
    print()
    print(classification_report(y, y_pred, target_names=['No compro', 'Compro']))
    return {'nombre': nombre, 'auc': auc, 'y_proba': y_proba, 'y_true': y}


res_train = evaluar_modelo(model, X_train, y_train, 'Train')
res_test  = evaluar_modelo(model, X_test,  y_test,  'Test')
res_val   = evaluar_modelo(model, X_val,   y_val,   'Validacion temporal')

In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colores = ['#145D38', '#9C247E', '#F79646']
for res, color in zip([res_train, res_test, res_val], colores):
    fpr, tpr, _ = roc_curve(res['y_true'], res['y_proba'])
    axes[0].plot(fpr, tpr, label=f"{res['nombre']} (AUC={res['auc']:.3f})", color=color)

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[0].set_title('Curvas ROC')
axes[0].set_xlabel('FPR')
axes[0].set_ylabel('TPR')
axes[0].legend()

ConfusionMatrixDisplay.from_estimator(
    model, X_val, y_val,
    display_labels=['No compro', 'Compro'],
    cmap='Greens',
    ax=axes[1],
)
axes[1].set_title('Matriz de Confusion — Validacion Temporal')

plt.tight_layout()
plt.show()